# Q-FRONTIER: Master Benchmark & Verification Notebook
### Intermountain Quantum Competition (IQC) 2026 — Brigham Young University (BYU)
**Project:** Q-FRONTIER (Quantum Financial Regime-Adaptive Optimization via Networked Trotterized Counterdiabatic Decomposition)  
**Sponsor Alignment:** Nasdaq, Quantum Rings, IonQ, Sandia National Laboratories, IBM Quantum, Classiq, qBraid, Aqora, BYU Physics  

---
### Abstract
This self-contained notebook reproduces the end-to-end mathematical formulation and verified quantitative benchmarks of **Q-FRONTIER**. It solves cardinality-constrained portfolio optimization on real Nasdaq equities by combining:
1. **Random Matrix Theory (RMT)** Marchenko-Pastur spectral denoising to partition assets into hardware-sized clusters ($n_c \le 4$).
2. A **4-Qubit 3-Layer Data-Reuploading Quantum Neural Network (QCL)** volatility sensor mapping primitive market signals to dynamic risk aversion $q_t$.
3. **Context-Aware Folding (CAF)** preserving cross-cluster covariance $\Sigma_{C_k, \bar{C}_k}$ with strict monotonic energy descent ($\Delta \mathcal{O} \le -\epsilon$) and a turnover continuity bonus ($\kappa$).
4. **Constrained Counterdiabatic QAOA (CCD-QAOA)** in fixed-Hamming-weight subspaces using Dicke states $|D_K^n\rangle$, complete-graph XY mixers, and 3-body Adiabatic Gauge Potential (AGP) driving ($H_{CD} \propto i[H_C, H_M^{XY}]$).
5. **SLSQP Continuous Weight Optimization**, a 5 bps net-of-cost walk-forward backtest, and **Sandia Volumetric Benchmarking** (Proctor et al., *Nature Physics* 2022).

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add backend to sys.path
sys.path.insert(0, os.path.abspath('../backend'))

from app.rmt_clustering import (
    DEFAULT_TICKERS,
    fetch_nasdaq_history,
    compute_expected_returns_and_covariance,
    marchenko_pastur_denoise,
    get_clusters
)
from app.qcl_volatility import QCLVolatilitySensor
from app.ccd_qaoa_solver import CCDQAOASolver
from app.caf_coordinator import CAFCoordinator
from app.slsqp_backtest import optimize_continuous_weights_slsqp, run_walk_forward_backtest
from app.hardware_benchmarks import generate_hardware_comparison, generate_sandia_volumetric_grid

print("Q-FRONTIER Modules Loaded Successfully!")

## 1. Stage 1: Nasdaq Ingestion & Marchenko-Pastur RMT Spectral Denoising
We pull 180-day price history for 12 liquid Nasdaq assets, compute Ledoit-Wolf shrinkage covariance, and filter out random noise modes where $\lambda < \lambda_+$.

In [ ]:
prices = fetch_nasdaq_history(DEFAULT_TICKERS, lookback_days=180)
mu, cov, tickers = compute_expected_returns_and_covariance(prices)
denoised_cov, denoised_corr, lambda_plus = marchenko_pastur_denoise(cov, len(prices), len(tickers))
clusters = get_clusters(denoised_corr, tickers, max_cluster_size=4)

print(f"Asset Universe: {len(tickers)} equities")
print(f"Marchenko-Pastur lambda+ cutoff: {lambda_plus:.4f}")
print(f"Hardware Clusters Partitioned: {len(clusters)} clusters (max 4 qubits each)")
for c in clusters:
    print(f"  Cluster {c['cluster_id']}: {c['tickers']}")

## 2. Stage 2: 4-Qubit Data-Reuploading QCL Volatility Sensor
Encodes four primitive signals: lagged return $r_{t-1}$, realized variance $\sigma^2_{t-1}$, downside leverage asymmetry, and rough-volatility increment $dV_t$.

In [ ]:
sensor = QCLVolatilitySensor()
market_proxy = prices.mean(axis=1).values
features = sensor.extract_primitive_features(market_proxy)
pred_vol, q_dynamic = sensor.predict_volatility(features)

print(f"Encoded Primitive Features: {features}")
print(f"QCL Predicted Annualized Volatility: {pred_vol * 100:.2f}%")
print(f"Dynamic Risk Aversion q_t: {q_dynamic:.4f}")

## 3. Stage 3: Constrained Counterdiabatic QAOA (CCD-QAOA) vs Baselines
Evaluates an exact 4-qubit, $K=2$ subproblem comparing Penalty-QAOA, XY-QAOA, and CCD-QAOA.

In [ ]:
solver = CCDQAOASolver(n_qubits=4, k_cardinality=2)
sub_cov = denoised_cov[:4, :4]
sub_mu = mu[:4]

res_p1 = solver.solve(sub_cov, sub_mu, q_risk=q_dynamic, depth=1)
res_p2 = solver.solve(sub_cov, sub_mu, q_risk=q_dynamic, depth=2)

print("=== SUBPROBLEM COMPARISON (n=4, K=2) ===")
print(f"Penalty-QAOA (p=1): AR = {res_p1['penalty_qaoa']['approximation_ratio']:.4f}, Feas = {res_p1['penalty_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p1['penalty_qaoa']['optimal_probability']*100:.2f}%")
print(f"Standard XY-QAOA (p=1): AR = {res_p1['xy_qaoa']['approximation_ratio']:.4f}, Feas = {res_p1['xy_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p1['xy_qaoa']['optimal_probability']*100:.2f}%")
print(f"CCD-QAOA (p=1):      AR = {res_p1['ccd_qaoa']['approximation_ratio']:.4f}, Feas = {res_p1['ccd_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p1['ccd_qaoa']['optimal_probability']*100:.2f}%")
print(f"CCD-QAOA (p=2):      AR = {res_p2['ccd_qaoa']['approximation_ratio']:.4f}, Feas = {res_p2['ccd_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p2['ccd_qaoa']['optimal_probability']*100:.2f}%")

## 4. Context-Aware Folding (CAF) Coordination Across All Clusters
Executes global context injection $v = \Sigma x$ with turnover continuity bonus $\kappa$.

In [ ]:
coordinator = CAFCoordinator(
    denoised_cov=denoised_cov,
    expected_returns=mu,
    clusters=clusters,
    q_risk=q_dynamic,
    continuity_bonus=0.05
)
caf_res = coordinator.run_caf_coordination(k_per_cluster=2, max_outer_iterations=2, depth=1)

selected_tickers = [tickers[i] for i in caf_res['selected_indices']]
print(f"Selected Optimal Assets: {selected_tickers}")
print(f"Static Decomposition Energy: {caf_res['static_energy']:.6f}")
print(f"Context-Aware Folding Energy: {caf_res['caf_energy']:.6f}")
print(f"Cross-Cluster Risk Reduction: {caf_res['risk_reduction_pct']:.2f}%")
print(f"Monotonic Energy Trajectory: {caf_res['energy_trajectory']}")

## 5. Stage 4: Continuous SLSQP Allocation & 5 bps Net-of-Cost Backtest

In [ ]:
weights, sharpe = optimize_continuous_weights_slsqp(caf_res['selected_indices'], mu, denoised_cov)
backtest = run_walk_forward_backtest(prices, caf_res['selected_indices'], transaction_cost_bps=5.0)

print(f"Continuous Optimized Sharpe Ratio: {sharpe:.2f}")
print(f"Backtest Final Portfolio Value: ${backtest['final_portfolio_value']} (vs ${backtest['benchmark_final_value']} Benchmark)")
print(f"Annualized Return: {backtest['annualized_return_pct']}% | Volatility: {backtest['annualized_volatility_pct']}%")
print(f"Net Sharpe Ratio: {backtest['net_sharpe_ratio']} | Max Drawdown: {backtest['max_drawdown_pct']}%")
print(f"Transaction Costs Paid: {backtest['transaction_costs_paid']:.3f}%")

## 6. Hardware Characterization: IonQ Trapped-Ion vs IBM Heavy-Hex & Sandia Volumetric Grid

In [ ]:
hw = generate_hardware_comparison(n_qubits=4)
print("=== HARDWARE TRANSPILATION AUDIT ===")
print(f"IonQ Trapped-Ion: Routing SWAPs = {hw['ionq_trapped_ion']['routing_swaps']} | 2Q Gates = {hw['ionq_trapped_ion']['two_qubit_gates']} | Depth = {hw['ionq_trapped_ion']['circuit_depth']}")
print(f"IBM Heavy-Hex:   Routing SWAPs = {hw['ibm_heavy_hex']['routing_swaps']} | 2Q Gates = {hw['ibm_heavy_hex']['two_qubit_gates']} | Depth = {hw['ibm_heavy_hex']['circuit_depth']}")

grid = generate_sandia_volumetric_grid()
print(f"\nSandia Volumetric Grid Computed: {len(grid)} cells across Widths [4, 6, 8, 12] and Depths [1, 2, 3, 4, 5]")